# Лабораторная работа 3

Цель работы:  
Сформировать начальный практический навык цикла разработки модели машинного обучения.

Инструкция:
- Запускайте ячейки по очереди, если не указаное иное действие.
- В конце выполните самостоятельное задание.

При первом запуске установите необходимые библиотеки.  
Раскомментируйте строку и запустите ячейку

In [ ]:
# !uv add scikit-learn

Используется результат лабораторной работы 2 в виде файла с БД
- labs/lab_02/data/db/data_lab02_prepared.db'

# 1. Подготовка данных и обучение базовой модели

Задача: Обучить модель, которая будет прогнозировать стоимость перевозки (Сумма в RUB)

## 1.1. Загрузка данных

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
from sqlalchemy import create_engine
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

In [ ]:
sns.set(context="paper")
RANDOM_STATE = 42
DB_NAME = 'data_lab02_prepared'
DB_PATH = f'../lab_02/data/db/{DB_NAME}.db'

In [ ]:
# Загрузка таблицы
engine = create_engine(f'sqlite:///{DB_PATH}')

query_advanced = f"""
    SELECT *
    FROM {DB_NAME}
"""
df = pd.read_sql_query(query_advanced, engine)

## 1.2. Разведочный анализ (EDA)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Строим гистограмму распределения стоимости перевозок
plt.figure(figsize=(8, 4))
sns.histplot(df['Сумма в RUB'], bins=30)
plt.title('Распределение стоимости перевозок (Сумма в RUB)')
plt.xlabel('Сумма в RUB')
plt.ylabel('Количество')
plt.show()

## 1.3. Формирование целевой переменной

Признак «Сумма в RUB» подходит в качестве целевой переменной, но его распределение имеет длинный правый хвост.  
Рекомендуется отсеять выбросы или обработать отдельно.

In [ ]:
# Посмотреть 99-й перцентиль => отсеять 1% самых больших значений
print(f"""99% всех значений меньше {df["Сумма в RUB"].quantile(0.99)}""")

In [ ]:
# Для задания отсеять 1%
# df =  # YOUR CODE

In [ ]:
df.shape

## 1.4. Создание новых признаков

In [ ]:
df["Дата операции"] = pd.to_datetime(df["Дата операции"], errors="coerce")
df["Дата время операции отправки"] = pd.to_datetime(
    df["Дата время операции отправки"],
    errors="coerce"
)

df["month"] = df["Дата операции"].dt.month
df["dayofweek"] = df["Дата операции"].dt.dayofweek
df["hour"] = df["Дата время операции отправки"].dt.hour

## 1.5 Выбор признаков

In [ ]:
# Выбираем только нужные нам колонки для прогноза
# Категориальные признаки
cat_cols = [
    "Классификатор перевозок",
    "Тип заказа",
    "Ранг отправки",
    "Операция",
    "Тип услуги",
    "Наименование плановой услуги предоставления",
    "Связка",
    "Тип клиента"
]
# Числовые признаки
num_cols = [
    "ДФЭ",
    "month",
    "dayofweek",
    "hour",
]
target_col = 'Сумма в RUB'

feature_cols = cat_cols + num_cols

In [ ]:
print("Количество уникальных значений в категориальных признаках:")
print(df[cat_cols].nunique().sort_values(ascending=False))

## 1.6. Разделение на обучающую и тестовую выборки

In [ ]:
X = df[feature_cols].copy()
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE
)

## 1.7. Предобработка категориальных признаков

Полезно посмотреть, сколько признаков было до one-hot кодирования

In [ ]:
print("Размер X_train до one-hot:", X_train.shape)
print("Размер X_test до one-hot:", X_test.shape)

Приведение категориальных признаков к строкам, иначе в обучении категория может быть, например, числом 1, 
а из JSON на этапе мониторинга модели придёт строка "1"

In [ ]:
X_train[cat_cols] = X_train[cat_cols].astype(str)
X_test[cat_cols] = X_test[cat_cols].astype(str)

Кодируем только категориальные колонки

In [ ]:
X_train = pd.get_dummies(X_train, columns=cat_cols, dtype=np.uint8)
X_test = pd.get_dummies(X_test, columns=cat_cols, dtype=np.uint8)

После этого категориальные колонки исчезнут, а вместо них появятся новые бинарные признаки вида:
- Клиент_Тип 1
- Клиент_Тип 2
- Тип заказа_Тип 1
- Тип заказа_Тип 2
- ...

При этом в train и test могут получиться разные наборы колонок. 
Например, какая-то категория есть только в test или только в train.

Выровняем состав признаков:
- оставляем только те колонки, которые есть в X_train;
- если в X_test есть колонка, которой не было в X_train, она удаляется;
- если в X_test нет какой-то колонки из X_train, она создаётся и заполняется нулями.

In [ ]:
X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

In [ ]:
print("Размер X_train после one-hot:", X_train.shape)
print("Размер X_test после one-hot:", X_test.shape)

## 1.8. Подготовка числовых признаков

In [ ]:
# Приводим числовые признаки к типу float
X_train[num_cols] = X_train[num_cols].astype(float)
X_test[num_cols] = X_test[num_cols].astype(float)

# Масштабируем числовые признаки
# scaler обучаем только на train, а test только преобразуем
scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

## 1.9. Обучение базовой модели

In [ ]:
# Обучаем линейную регрессию
model = LinearRegression()
model.fit(X_train, y_train)
print("Done!")

In [ ]:
# Обучаем модель
# model = # YOUR CODE
# model.fit(X_train, y_train)
# print("Done!")

Оценка качества

In [ ]:
# Делаем прогноз
predictions = model.predict(X_test)

# Считаем ошибку
mae = mean_absolute_error(y_test, predictions)

print(f"Средняя абсолютная ошибка (MAE): {mae:.0f} рублей.")
print("Это означает, что в среднем модель ошибается на эту сумму при прогнозе.")

Сохранение модели

In [ ]:
# Создаём папку для результатов
timestamp = datetime.now().strftime("%Y%m%d_%H-%M-%S")
output_dir = f"results/{timestamp}"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Сохраняем модель
joblib.dump(model, os.path.join(output_dir, "model.joblib"))
joblib.dump(scaler, os.path.join(output_dir, "scaler.joblib"))
joblib.dump(list(X_train.columns), os.path.join(output_dir, "feature_columns.joblib"))
print(f"Данные сохранены в: {output_dir}")

## 2. Визуализация результатов предсказания

In [ ]:
# Строим график факт vs прогноз
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=predictions, alpha=0.3, s=15, label='Заявки')

# Рисуем линию идеального прогноза
min_val = min(y_test.min(), predictions.min())
max_val = max(y_test.max(), predictions.max())
plt.plot([min_val, max_val], [min_val, max_val], color='darkblue', label='Идеальный прогноз')

plt.xlabel('Фактическая стоимость (RUB)')
plt.ylabel('Предсказанная стоимость (RUB)')
plt.title('Сравнение фактической и предсказанной стоимости')
plt.legend()

# Сохраняем график
plot_path = os.path.join(output_dir, f"fact_vs_prediction.png")
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
print(f"График сохранён: {plot_path}")

plt.show()

In [ ]:
# Считаем ошибки для каждой заявки
errors = y_test - predictions

plt.figure(figsize=(10, 5))
sns.histplot(errors, bins=40, color='tab:orange')

# Добавляем вертикальную линию на нуле
plt.axvline(x=0, color='darkred', linestyle='--', label='Нулевая ошибка')
plt.xlabel('Ошибка прогноза (RUB)')
plt.ylabel('Количество')
plt.title('Распределение ошибок модели')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')

# Добавляем статистику на график
stats_text = (
    f"Средняя ошибка: {errors.mean():.0f} руб.\n"
    f"Медианная ошибка: {errors.median():.0f} руб.\n"
    f"Максимальная ошибка: {errors.abs().max():.0f} руб."
)
plt.text(0.95, 0.95, stats_text,
         transform=plt.gca().transAxes,
         fontsize=10,
         verticalalignment='top',
         horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Сохраняем график
plot_path = os.path.join(output_dir, f"errors_{timestamp}.png")
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
print(f"График сохранён: {plot_path}")

plt.show()

In [ ]:
print(f"Средняя ошибка: {errors.mean():.0f} руб.")
print(f"Медианная ошибка: {errors.median():.0f} руб.")
print(f"Максимальная ошибка: {errors.abs().max():.0f} руб.")

# 3. Самостоятельная работа

Задание:
1. Запустите ячейки и сохраните результат.
2. Повторите запуск, но на этапе 1.3 отфильтруйте из данных 1% самых больших значений. Сравните полученный результат.
3. Повторите запуск, но пропустите этап "Подготовка числовых признаков". Сравните полученный результат.
4. Повторите запуск, но замените модель на DecisionTreeRegressor. Используйте код `DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE)`. Сравните полученный результат.  
5. Повторите запуск с DecisionTreeRegressor, но пропустите этап "Подготовка числовых признаков". Сравните полученный результат.

Вопросы:
- Для заданий 2-5 попробуйте объяснить, чем обусловлены наблюдаемые изменения.
- Какие основные этапы подготовки признаков пропущены и почему?
- Для какой из моделей важна подготовка числовых признаков? Укажите изменения метрик для обеих моделей.

Подсказка:
- https://deepmachinelearning.ru/docs/Machine-learning/Data-preprocessing/Feature-normalization
- https://deepmachinelearning.ru/docs/Machine-learning/Data-preprocessing/Data-imputation
